# Notebook 13 — Bayesian Statistics Intuition

**Module:** 03 — Statistics and Probability  
**Notebook:** 13 of 18  
**Prerequisites:** NB03, NB05  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Many European computational biology groups (EMBL-EBI, IMPRS, Wellcome Sanger)
work in a Bayesian framework — Bayesian graphical models, variational inference,
probabilistic single-cell analysis. Understanding the Bayesian update cycle — prior,
likelihood, posterior — is essential for reading their papers and for a competitive
Track B application.

---
## Step 2 — Intuition

**Frequentist:** data are random; parameters are fixed unknown constants.
**Bayesian:** parameters are random variables with distributions; data are observed and fixed.

The Bayesian update: start with a prior belief $P(\theta)$ about a parameter.
Observe data $x$. Update to a posterior belief $P(\theta|x)$ using Bayes' theorem:
$$P(\theta|x) \propto P(x|\theta) \cdot P(\theta)$$

The posterior is a compromise between your prior belief and what the data say.

---
## Step 3 — Biological Background

**Allele frequency estimation from sequencing data:**
- You sequenced a population and observed $k$ reads showing the alternate allele out of $n$ total
- **Frequentist:** $\hat{p} = k/n$ (MLE). Confidence interval from binomial formula.
- **Bayesian:** Prior: $p \sim \text{Beta}(\alpha, \beta)$ (allele frequency in (0,1)).
  Likelihood: $k \sim \text{Binomial}(n, p)$.
  Posterior: $p|k,n \sim \text{Beta}(\alpha + k, \beta + n - k)$ — conjugate prior!
- **Credible interval:** 95% central interval of the posterior — directly interpretable
  as 'the parameter is in this range with 95% probability.'

**DESeq2 and empirical Bayes:**
DESeq2's shrinkage of fold change estimates is an empirical Bayes approach — it
shrinks noisy estimates (from low-count genes) toward the genome-wide average,
using the population of genes as a prior.

---
## Step 4 — Mathematical Explanation

**Bayes' theorem for continuous parameters:**
$$P(\theta|x) = \frac{P(x|\theta) \cdot P(\theta)}{P(x)} \propto P(x|\theta) \cdot P(\theta)$$

**Beta-Binomial conjugate pair:**
- Prior: $\theta \sim \text{Beta}(\alpha_0, \beta_0)$
- Likelihood: $k \sim \text{Binomial}(n, \theta)$
- Posterior: $\theta|k \sim \text{Beta}(\alpha_0 + k, \beta_0 + n - k)$

The posterior mean:
$$E[\theta|k] = \frac{\alpha_0 + k}{\alpha_0 + \beta_0 + n}$$
which is a weighted average of the prior mean $\alpha_0/(\alpha_0+\beta_0)$
and the MLE $k/n$. With more data ($n \to \infty$), the likelihood dominates.

**95% Bayesian credible interval:**
The interval $[a, b]$ such that $P(\theta \in [a,b]|x) = 0.95$.
NOT the same as a frequentist confidence interval.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Beta-Binomial: allele frequency estimation
def bayesian_allele_freq(k: int, n: int, alpha0: float = 1.0, beta0: float = 1.0):
    """
    Bayesian estimation of allele frequency p using Beta-Binomial model.

    Parameters
    ----------
    k : observed alternate allele reads
    n : total reads
    alpha0, beta0 : Beta prior parameters (default: uniform prior)

    Returns
    -------
    dict with posterior parameters, mean, MAP, 95% credible interval
    """
    alpha_post = alpha0 + k
    beta_post  = beta0  + (n - k)
    posterior  = stats.beta(alpha_post, beta_post)
    post_mean  = alpha_post / (alpha_post + beta_post)
    post_mode  = (alpha_post - 1) / (alpha_post + beta_post - 2) if alpha_post > 1 else 0
    ci_95 = posterior.ppf([0.025, 0.975])
    return dict(alpha_post=alpha_post, beta_post=beta_post,
                posterior=posterior, post_mean=post_mean, post_mode=post_mode,
                ci_95=ci_95, mle=k/n if n > 0 else 0)

# Three scenarios: rare allele, moderate, common
scenarios = [(2, 20, "Sparse data"), (10, 100, "Moderate"), (100, 1000, "Dense sequencing")]
print(f"{'Scenario':<22} {'k/n':>6} {'MLE':>8} {'Post.mean':>10} {'95% CI'}")
print("-" * 65)
for k, n, label in scenarios:
    res = bayesian_allele_freq(k, n)
    print(f"  {label:<20} {k}/{n}   {res['mle']:>6.3f} {res['post_mean']:>10.3f}   [{res['ci_95'][0]:.3f}, {res['ci_95'][1]:.3f}]")

In [ ]:
# Cell 6.2 — Sequential updating: watching the posterior sharpen with more data
rng = np.random.default_rng(42)
true_p = 0.3  # true allele frequency

# Start with uniform prior
alpha0, beta0 = 1.0, 1.0
results_seq = []

for n_reads in [0, 5, 20, 100, 500]:
    if n_reads == 0:
        res = dict(alpha_post=alpha0, beta_post=beta0,
                   post_mean=0.5, ci_95=[0.025, 0.975], mle=None)
    else:
        k_obs = rng.binomial(n_reads, true_p)
        res = bayesian_allele_freq(k_obs, n_reads, alpha0, beta0)
        res['n'] = n_reads; res['k'] = k_obs
    results_seq.append((n_reads, res))

print(f"Sequential updates (true p = {true_p}):")
for n_reads, res in results_seq:
    ci = res['ci_95']
    print(f"  n={n_reads:>4}: posterior mean={res['post_mean']:.3f}  95% CI=[{ci[0]:.3f}, {ci[1]:.3f}]")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Prior, likelihood, and posterior for Beta-Binomial
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Prior → Posterior with increasing data
ax = axes[0]
p_range = np.linspace(0.001, 0.999, 500)
ax.plot(p_range, stats.beta(1, 1).pdf(p_range), 'gray', ls='--', lw=1.5, label="Prior: Uniform(0,1)")
colors_seq = ["steelblue", "green", "orange", "tomato"]
for (n_reads, res), color in zip(results_seq[1:], colors_seq):
    post = stats.beta(res['alpha_post'], res['beta_post'])
    ax.plot(p_range, post.pdf(p_range), color=color, lw=1.5,
            label=f"n={n_reads}: k={res.get('k','?')}")
ax.axvline(true_p, color='black', ls=':', lw=1, label=f"True p={true_p}")
ax.set_xlabel("Allele frequency p"); ax.set_ylabel("Posterior density")
ax.set_title("Bayesian updating: posterior sharpens with data")
ax.legend(fontsize=8)

# Panel 2: Credible interval vs. frequentist CI width vs. n
ax = axes[1]
n_vals = np.arange(5, 501, 5)
bayes_width, freq_width = [], []
for n in n_vals:
    k = int(n * true_p)
    r = bayesian_allele_freq(k, n)
    bayes_width.append(r['ci_95'][1] - r['ci_95'][0])
    # Frequentist Wilson CI
    p_hat = k/n
    se = np.sqrt(p_hat*(1-p_hat)/n)
    freq_width.append(2 * 1.96 * se)
ax.plot(n_vals, bayes_width, color="steelblue", lw=2, label="Bayesian 95% credible interval")
ax.plot(n_vals, freq_width, color="tomato", ls="--", lw=2, label="Frequentist 95% CI")
ax.set_xlabel("n reads"); ax.set_ylabel("Interval width")
ax.set_title("Credible interval narrows with more data")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `bayesian_allele_freq()` from scratch. Verify the posterior mean formula
   analytically: show that $E[p|k,n] = (\alpha_0+k)/(\alpha_0+\beta_0+n)$.
2. Use an informative prior Beta(5, 15) (encoding a belief that the allele is rare,
   prior mean=0.25). Observe k=8 out of n=20. Compare the posterior mean to the
   uninformative (uniform) prior posterior. Which prior dominates? When would you
   prefer an informative prior?
3. What is the difference between a Bayesian 95% credible interval and a frequentist
   95% confidence interval? Write two sentences — make the distinction precise.
4. DESeq2's shrinkage estimator is described as 'empirical Bayes.' What does this mean?
   How does it relate to the Beta-Binomial model in this notebook?

---
## Quiz — Active Recall

1. Write Bayes' theorem for continuous parameters.
2. What are 'prior', 'likelihood', and 'posterior'? Define each in one sentence.
3. What is a conjugate prior? Why is Beta-Binomial conjugate?
4. What is a credible interval? How does it differ from a confidence interval?
5. Why does the posterior approach the likelihood as n → ∞?

---
## Reflection

**Date completed:** ____________________

> *[Could you explain the Bayesian update cycle to a frequentist colleague? What is the key conceptual difference?]*

---
*Next: `14_resampling_bootstrap_permutation.ipynb`*